# E02 — découpage train / dev / test

Le gain du trigramme est-il réel, ou le modèle mémorise-t-il le corpus ?
Jusqu'ici tout est évalué sur les données d'entraînement, ce qui ne dit rien
de la capacité à généraliser.

In [1]:
words = open('names.txt', 'r').read().splitlines()

In [2]:
import random
import torch

## Découpage

On mélange et on découpe des **mots entiers**.
Un même prénom à cheval sur train et dev mettrait des morceaux déjà vus
dans le dev, qui ne mesurerait plus rien.

La graine fixée rend le découpage reproductible ; la copie `words[:]`
évite de mélanger la liste d'origine en place.

In [3]:
random.seed(42)
ws = words[:] # copie, on ne mélange pas words en place
random.shuffle(ws)
n1, n2 = int(0.8 * len(ws)), int(0.9 * len(ws))
train, dev, test = ws[:n1], ws[n1:n2], ws[n2:]
print(len(train), len(dev), len(test))

25626 3203 3204


In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

## Bigramme

In [5]:
def bigrams(ws):
    out = []
    for w in ws:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            out.append((stoi[ch1], stoi[ch2]))
    return torch.tensor(out)

bi_train, bi_dev, bi_test = bigrams(train), bigrams(dev), bigrams(test)

In [6]:
def counts2(bi):
    N = torch.zeros((27, 27), dtype=torch.int32)
    for i, j in bi:
        N[i, j] += 1
    return N

N2 = counts2(bi_train)

In [7]:
def proba2(N, smooth):
    P = (N + smooth).float()
    return P / P.sum(1, keepdim=True)

def loss2(P, bi):
    return -P[bi[:, 0], bi[:, 1]].log().mean().item()

for sm in [1, 0.001]:
    P2 = proba2(N2, sm)
    print(f'bigramme sm={sm}: train {loss2(P2, bi_train):.4f}  dev {loss2(P2, bi_dev):.4f}  test {loss2(P2, bi_test):.4f}')

bigramme sm=1: train 2.4548  dev 2.4533  test 2.4584
bigramme sm=0.001: train 2.4541  dev 2.4540  test 2.4587


## Trigramme

In [8]:
def trigrams(ws):
    out = []
    for w in ws:
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            out.append((stoi[ch1], stoi[ch2], stoi[ch3]))
    return torch.tensor(out)

tri_train, tri_dev, tri_test = trigrams(train), trigrams(dev), trigrams(test)

In [9]:
def counts3(tri):
    N = torch.zeros((27, 27, 27), dtype=torch.int32)
    for i, j, k in tri:
        N[i, j, k] += 1
    return N

N3 = counts3(tri_train)

In [10]:
def proba3(N, smooth):
    P = (N + smooth).float()
    return P / P.sum(2, keepdim=True)

def loss3(P, tri):
    return -P[tri[:, 0], tri[:, 1], tri[:, 2]].log().mean().item()

for sm in [1, 0.001]:
    P3 = proba3(N3, sm)
    print(f'trigramme sm={sm}: train {loss3(P3, tri_train):.4f}  dev {loss3(P3, tri_dev):.4f}  test {loss3(P3, tri_test):.4f}')

trigramme sm=1: train 2.2156  dev 2.2365  test 2.2373
trigramme sm=0.001: train 2.1827  dev 2.2447  test 2.2487
